In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")
from busqueda import agrupar, coincidencias, compuerta, dentro, disparos, duracion, far, interseccion, segmentos_de_analisis, tiempo_fondo, tiempo_virtual
from datos import cargar_archivo, gps_evento, segmentos
from filtro_adaptado import chi2, psd_en, serie_snr, snr_reponderado, transformada
from plantillas import generador

In [ ]:
EVENTO = "GW150914"
DATOS = Path("../datos")
CONJUNTO = "O1"
ARCHIVOS = [1126256640, 1126260736]
DURACION_ARCHIVO = 4096
FLAGS = ["DATA", "CBC_CAT1", "NO_CBC_HW_INJ"]
FLAG_VETO = "CBC_CAT2"
RECORTE_BLOQUE = 8
DURACION_SEGMENTO = 256
BORDE_INICIAL = 16
BORDE_FINAL = 4
DURACION_PSD = 512
DURACION_FFT = 4
F_MIN = 20
F_MAX = 1024
FS_SALIDA = 2048
TANDA = 25
SNR_UMBRAL = 5.0
VENTANA_DISPAROS = 1
BANDAS_CHI2 = 16
VENTANA_COINCIDENCIA = 0.012
PASO_SLIDES = 0.1
VENTANA_AGRUPAR = 1
VENTANA_EVENTO = 1
COMPUERTA_UMBRAL = 50
COMPUERTA_ANCHO = 0.25
COMPUERTA_TRANSICION = 0.25
UMBRALES_RED = [7, 8, 9, 10]
MOSTRAR = 10
AÑO = 365.25 * 86400

In [ ]:
t0 = gps_evento(EVENTO, DATOS)
inicio, fin = ARCHIVOS[0], ARCHIVOS[-1] + DURACION_ARCHIVO
buenos = [(inicio, fin)]
for ifo in ["H1", "L1"]:
    for flag in FLAGS:
        buenos = interseccion(buenos, segmentos(f"{ifo}_{flag}", inicio, fin, DATOS))
bloques = [(a + RECORTE_BLOQUE, b - RECORTE_BLOQUE) for a, b in buenos if b - a >= DURACION_SEGMENTO + 2 * RECORTE_BLOQUE]
analisis = [(a + BORDE_INICIAL, b - BORDE_FINAL) for a, b in bloques]
for ifo in ["H1", "L1"]:
    analisis = interseccion(analisis, segmentos(f"{ifo}_{FLAG_VETO}", inicio, fin, DATOS))
tiempo_analizado = duracion(analisis)
print(f"H1 y L1 con {', '.join(FLAGS)}: {duracion(buenos)} s, de {buenos[0][0] - t0:+.0f} a {buenos[-1][1] - t0:+.0f} s desde t0")
print(f"Tiempo analizado (sin los {RECORTE_BLOQUE} s de cada extremo, sin los bordes de los segmentos y sin los vetos {FLAG_VETO}): {tiempo_analizado} s en {len(analisis)} tramos")

In [ ]:
series = {ifo: [cargar_archivo(ifo, a, CONJUNTO, DATOS) for a in ARCHIVOS] for ifo in ["H1", "L1"]}
fs = int(series["H1"][0].sample_rate.value)
crudos = {ifo: np.concatenate([serie.value for serie in lista]) for ifo, lista in series.items()}
n = DURACION_SEGMENTO * fs
df = fs / n
n_salida = n * FS_SALIDA // fs
indices = np.arange(int(F_MIN / df), int(F_MAX / df) + 1)
frecuencias = np.arange(n // 2 + 1) * df


def muestras(a, b):
    return slice(int((a - inicio) * fs), int((b - inicio) * fs))

In [ ]:
reloj = time.perf_counter()
tramos = []
datos_f = {"H1": [], "L1": []}
psd = {"H1": [], "L1": []}
compuertas = {"H1": set(), "L1": set()}
for a, b in bloques:
    for s, v0, v1 in segmentos_de_analisis(a, b, DURACION_SEGMENTO, BORDE_INICIAL, BORDE_FINAL):
        tramos.append((s, v0, v1))
        centro = s + DURACION_SEGMENTO / 2
        for ifo, x in crudos.items():
            limpio, centros = compuerta(x[muestras(s, s + DURACION_SEGMENTO)], fs, COMPUERTA_UMBRAL, COMPUERTA_ANCHO, COMPUERTA_TRANSICION, np.fft.fft)
            compuertas[ifo].update(round(s + c - t0, 2) for c in centros)
            datos_f[ifo].append(transformada(limpio, fs, fs // 2, np.fft.fft))
            psd[ifo].append(psd_en(frecuencias, x[muestras(max(a, centro - DURACION_PSD / 2), min(b, centro + DURACION_PSD / 2))], fs, DURACION_FFT, np.fft.fft))
print(f"{len(tramos)} segmentos de {DURACION_SEGMENTO} s por detector, preparados en {time.perf_counter() - reloj:.0f} s")
for ifo in compuertas:
    print(f"Compuertas en {ifo} (s desde t0): {sorted(compuertas[ifo])}")

In [ ]:
banco = np.load(DATOS / "banco.npz")["banco"]
plantilla = generador(indices * df, F_MIN)
encontrados = {"H1": [], "L1": []}
perfil = np.zeros(DURACION_SEGMENTO)
cuantos = 0
reloj = time.perf_counter()
for a in range(0, len(banco), TANDA):
    plantillas = np.array([plantilla(*p) for p in banco[a : a + TANDA]])
    for k, (s, v0, v1) in enumerate(tramos):
        desde = int((v0 - s) * FS_SALIDA)
        hasta = int((v1 - s) * FS_SALIDA)
        for ifo in ["H1", "L1"]:
            rho = serie_snr(datos_f[ifo][k], plantillas, psd[ifo][k], df, n_salida, indices, np.fft.ifft)
            if not s <= t0 < s + DURACION_SEGMENTO:
                perfil += (rho**2).reshape(len(plantillas), DURACION_SEGMENTO, FS_SALIDA).mean(axis=2).sum(axis=0)
                cuantos += len(plantillas)
            p, m, valores = disparos(rho[:, desde:hasta], VENTANA_DISPAROS * FS_SALIDA, SNR_UMBRAL)
            m = m + desde
            for q in np.unique(p):
                cuales = p == q
                valores_chi2 = chi2(datos_f[ifo][k], plantillas[q], psd[ifo][k], df, n_salida, indices, m[cuales], BANDAS_CHI2)
                encontrados[ifo].append(np.column_stack([s + m[cuales] / FS_SALIDA, np.full(cuales.sum(), a + q), valores[cuales], valores_chi2]))
    if (a // TANDA) % 5 == 4 or a + TANDA >= len(banco):
        print(f"{min(a + TANDA, len(banco))}/{len(banco)} plantillas, {time.perf_counter() - reloj:.0f} s")
perfil /= cuantos

In [ ]:
print(f"ρ² medio en la parte válida: {perfil[BORDE_INICIAL:-BORDE_FINAL].mean():.3f} (ruido gaussiano con la PSD de {DURACION_PSD} s: ≈ 2 × 1.016 = 2.03)")
plt.semilogy(np.arange(DURACION_SEGMENTO) + 0.5, perfil)
plt.axvspan(BORDE_INICIAL, DURACION_SEGMENTO - BORDE_FINAL, alpha=0.15, label="parte válida")
plt.axhline(2, color="k", linestyle="--", label="ruido gaussiano")
plt.xlabel("Segundo dentro del segmento")
plt.ylabel("ρ² medio")
plt.legend();

In [ ]:
grados = 2 * BANDAS_CHI2 - 2
tabla = {}
for ifo in ["H1", "L1"]:
    d = np.concatenate(encontrados[ifo])
    vetados = ~dentro(d[:, 0], analisis)
    d = d[~vetados]
    tabla[ifo] = {"t": d[:, 0], "plantilla": d[:, 1].astype(int), "rho": d[:, 2], "chi2": d[:, 3], "rep": snr_reponderado(d[:, 2], d[:, 3], BANDAS_CHI2)}
    print(f"{ifo}: {len(d)} disparos con SNR ≥ {SNR_UMBRAL} ({vetados.sum()} fuera del tiempo analizado); χ²_r mediano {np.median(d[:, 3]) / grados:.2f}; con el SNR reponderado ≥ {SNR_UMBRAL} quedan {(tabla[ifo]['rep'] >= SNR_UMBRAL).sum()}")

In [ ]:
tau = {ifo: tiempo_virtual(tabla[ifo]["t"], analisis) for ifo in tabla}
i, j, k = coincidencias(tau["H1"], tabla["H1"]["plantilla"], tau["L1"], tabla["L1"]["plantilla"], VENTANA_COINCIDENCIA, PASO_SLIDES)
reales = (k != 0) | (np.abs(tabla["H1"]["t"][i] - tabla["L1"]["t"][j]) <= VENTANA_COINCIDENCIA)
i, j, k = i[reales], j[reales], k[reales]
red = np.hypot(tabla["H1"]["rep"][i], tabla["L1"]["rep"][j])
cerca_evento = (np.abs(tabla["H1"]["t"][i] - t0) < VENTANA_EVENTO) | (np.abs(tabla["L1"]["t"][j] - t0) < VENTANA_EVENTO)
grupos = agrupar(k, tau["H1"][i], red, VENTANA_AGRUPAR)
lejos = np.flatnonzero(~cerca_evento)
grupos_exclusivos = lejos[agrupar(k[lejos], tau["H1"][i][lejos], red[lejos], VENTANA_AGRUPAR)]
primer_plano = grupos[k[grupos] == 0]
fondo = grupos[k[grupos] != 0]
fondo_exclusivo = grupos_exclusivos[k[grupos_exclusivos] != 0]
tiempo_del_fondo = tiempo_fondo(tiempo_analizado, PASO_SLIDES)
print(f"{len(i)} coincidencias (misma plantilla, ±{1000 * VENTANA_COINCIDENCIA:.0f} ms) en {int(tiempo_analizado / PASO_SLIDES)} desplazamientos de {PASO_SLIDES} s a cada lado")
print(f"Agrupadas en ventanas de {VENTANA_AGRUPAR} s: primer plano {len(primer_plano)}, fondo {len(fondo)} (exclusivo: {len(fondo_exclusivo)})")
print(f"Tiempo de fondo: {tiempo_del_fondo:.3e} s = {tiempo_del_fondo / AÑO:.1f} años")

In [ ]:
def ifar(valor, fondo_usado):
    tasa = far(red[fondo_usado], tiempo_del_fondo, valor)
    if tasa == 0:
        return f"> {tiempo_del_fondo / AÑO:.1f} años"
    return f"{1 / tasa / AÑO:.3g} años"


for c in primer_plano[np.argsort(-red[primer_plano])][:MOSTRAR]:
    h1, l1 = i[c], j[c]
    print(f"t0 {tabla['H1']['t'][h1] - t0:+9.3f} s, Mc {banco[tabla['H1']['plantilla'][h1], 0]:5.1f}: H1 {tabla['H1']['rho'][h1]:5.1f} (χ²_r {tabla['H1']['chi2'][h1] / grados:5.2f}) → {tabla['H1']['rep'][h1]:4.1f}, L1 {tabla['L1']['rho'][l1]:5.1f} (χ²_r {tabla['L1']['chi2'][l1] / grados:5.2f}) → {tabla['L1']['rep'][l1]:4.1f}; red {red[c]:5.2f}, IFAR inclusivo {ifar(red[c], fondo)}, exclusivo {ifar(red[c], fondo_exclusivo)}")
print(f"Fondo más fuerte: {red[fondo].max():.2f} (exclusivo: {red[fondo_exclusivo].max():.2f})")

In [ ]:
sin_evento = primer_plano[~cerca_evento[primer_plano]]
for umbral in UMBRALES_RED:
    esperado = far(red[fondo_exclusivo], tiempo_del_fondo, umbral) * tiempo_analizado
    print(f"Red ≥ {umbral}: primer plano sin {EVENTO} {np.sum(red[sin_evento] >= umbral)}, esperado por el fondo {esperado:.2f} ± {np.sqrt(esperado):.2f}")

In [ ]:
valores = np.sort(red[primer_plano])[::-1]
x = np.linspace(red[fondo_exclusivo].min(), valores.max() + 1, 400)
esperado = far(red[fondo_exclusivo], tiempo_del_fondo, x) * tiempo_analizado
plt.fill_between(x, np.maximum(esperado - 2 * np.sqrt(esperado), 1e-3), esperado + 2 * np.sqrt(esperado), color="gray", alpha=0.3, label="fondo ± 2σ")
plt.semilogy(x, esperado, "k--", label="fondo (time slides, exclusivo)")
plt.step(valores, np.arange(1, len(valores) + 1), where="post", label="primer plano")
plt.ylim(1e-3, None)
plt.xlabel("SNR de red reponderado")
plt.ylabel("Coincidencias por encima")
plt.legend();

In [ ]:
evento = primer_plano[np.argmax(red[primer_plano] * cerca_evento[primer_plano])]
print(f"{EVENTO}: SNR de red reponderado {red[evento]:.2f}, en t0 {tabla['H1']['t'][i[evento]] - t0:+.3f} s, L1 − H1 {1000 * (tabla['L1']['t'][j[evento]] - tabla['H1']['t'][i[evento]]):+.1f} ms")
print(f"IFAR inclusivo {ifar(red[evento], fondo)}; exclusivo {ifar(red[evento], fondo_exclusivo)} (con {tiempo_analizado / 3600:.1f} h de datos)")